# Hansen (2015) Replication and RD-Compatible Causal Machine Learning Extension

This script preserves the reduced Table 2 balance replication, adds local-linear
regression-discontinuity estimates for four-year DUI recidivism, and implements
machine learning only inside the regression-discontinuity design.

In [1]:
# Import standard-library modules for file paths, version reporting, and warnings.
from __future__ import annotations

import importlib.util
import json
import platform
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

# Import numerical and data-management packages used throughout the analysis.
import numpy as np
import pandas as pd

# Configure matplotlib for file-based figure creation in scripts and headless environments.
import matplotlib
matplotlib.use("Agg")

# Import matplotlib after selecting the non-interactive backend.
import matplotlib.pyplot as plt

# Import statsmodels for transparent local-polynomial RD estimation and clustered inference.
import statsmodels.api as sm

# Import statistical utilities for p-values and confidence intervals.
from scipy import stats

# Import LightGBM for shallow, regularized boosted-tree nuisance models.
from lightgbm import LGBMClassifier

# Import scikit-learn tools for group-aware cross-fitting and prediction diagnostics.
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold

# Suppress only repetitive LightGBM feature-name warnings while preserving substantive warnings.
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Set one project-wide random seed so every stochastic result is reproducible.
RANDOM_SEED = 610

# Define the two statutory BAC thresholds studied in Hansen (2015).
THRESHOLDS = {
    "DUI threshold": 0.08,
    "Aggravated DUI threshold": 0.15,
}

# Define the baseline local window used in the existing replication and presentation.
BASELINE_BANDWIDTH = 0.05

# Define bandwidths used to test whether the estimated discontinuity is sensitive to the local window.
SENSITIVITY_BANDWIDTHS = (0.02, 0.03, 0.04, 0.05)

# Define predetermined covariates that are measured before punishment is assigned.
PREDETERMINED_COVARIATES = ("aged", "male", "white", "acc")

# Define the number of group-aware folds used for honest out-of-fold ML predictions.
ML_CROSS_FOLDS = 3

# Define the number of BAC-bin bootstrap replications used for exploratory CATE uncertainty.
ML_BOOTSTRAP_REPLICATIONS = 20

# Define shallow LightGBM settings to reduce overfitting with a small covariate set.
LIGHTGBM_PARAMETERS: dict[str, Any] = {
    "objective": "binary",
    "n_estimators": 30,
    "learning_rate": 0.05,
    "num_leaves": 7,
    "max_depth": 3,
    "min_child_samples": 200,
    "reg_lambda": 1.0,
    "verbosity": -1,
    "n_jobs": 2,
}

## 1. Project setup and reproducibility

In [2]:
@dataclass(frozen=True)
class ProjectPaths:
    """Store all project input and output paths in one immutable object."""

    root: Path
    data_file: Path
    table_directory: Path
    figure_directory: Path
    code_directory: Path




def maybe_upload_dataset_in_colab(starting_path: Path | None = None) -> None:
    """Offer a Colab upload only when the course dataset is missing in a Colab session."""

    # Return immediately in local Python environments so Google-specific imports remain optional.
    if importlib.util.find_spec("google.colab") is None:
        return

    # Search for a likely repository root using the current directory and its parents.
    start = (starting_path or Path.cwd()).resolve()
    candidate_roots = [candidate for candidate in (start, *start.parents) if (candidate / "code").exists()]
    project_root = candidate_roots[0] if candidate_roots else start
    expected_data_file = project_root / "data" / "modified_hansen_data.csv"

    # Skip the upload dialog when the repository already contains the required dataset.
    if expected_data_file.exists():
        return

    # Import the Colab file picker only inside an actual Colab environment.
    from google.colab import files

    # Explain exactly which file the user must select before opening the upload dialog.
    print("Select modified_hansen_data.csv so the notebook can place it in the repository data folder.")
    uploaded_files = files.upload()

    # Stop with a clear error when the required filename was not selected.
    if "modified_hansen_data.csv" not in uploaded_files:
        raise FileNotFoundError("The Colab upload did not include modified_hansen_data.csv")

    # Create the data directory and write the uploaded bytes to the expected relative path.
    expected_data_file.parent.mkdir(parents=True, exist_ok=True)
    expected_data_file.write_bytes(uploaded_files["modified_hansen_data.csv"])
    print(f"Saved uploaded dataset to: {expected_data_file}")

def locate_project_root(starting_path: Path | None = None) -> Path:
    """Locate the repository root by searching for the expected data directory."""

    # Start from the supplied path, or from the current working directory when no path is supplied.
    start = (starting_path or Path.cwd()).resolve()

    # Search the current directory and each parent for the expected dataset path.
    for candidate in (start, *start.parents):
        expected_data_file = candidate / "data" / "modified_hansen_data.csv"
        if expected_data_file.exists():
            return candidate

    # Use the script location as a second search route when the code runs from another directory.
    if "__file__" in globals():
        script_directory = Path(__file__).resolve().parent
        for candidate in (script_directory, *script_directory.parents):
            expected_data_file = candidate / "data" / "modified_hansen_data.csv"
            if expected_data_file.exists():
                return candidate

    # Raise a clear error instead of silently reading an unintended file.
    raise FileNotFoundError(
        "Could not locate data/modified_hansen_data.csv. Run the code from the repository "
        "or place the dataset in the repository's data folder."
    )


def build_project_paths(project_root: Path) -> ProjectPaths:
    """Create output directories and return the paths used by the analysis."""

    # Define the input dataset using a path relative to the repository root.
    data_file = project_root / "data" / "modified_hansen_data.csv"

    # Define separate table and figure folders so outputs remain organized.
    table_directory = project_root / "output" / "tables"
    figure_directory = project_root / "output" / "figures"
    code_directory = project_root / "code"

    # Create missing output directories without overwriting existing files.
    table_directory.mkdir(parents=True, exist_ok=True)
    figure_directory.mkdir(parents=True, exist_ok=True)
    code_directory.mkdir(parents=True, exist_ok=True)

    # Return the complete path configuration for reuse in every function.
    return ProjectPaths(
        root=project_root,
        data_file=data_file,
        table_directory=table_directory,
        figure_directory=figure_directory,
        code_directory=code_directory,
    )


def environment_versions() -> pd.DataFrame:
    """Return the software versions needed to reproduce the analysis."""

    # Import package namespaces locally so their installed versions can be recorded.
    import lightgbm
    import matplotlib as matplotlib_package
    import sklearn
    import statsmodels

    # Store one row per software dependency for a machine-readable version table.
    return pd.DataFrame(
        [
            {"software": "Python", "version": platform.python_version()},
            {"software": "pandas", "version": pd.__version__},
            {"software": "numpy", "version": np.__version__},
            {"software": "statsmodels", "version": statsmodels.__version__},
            {"software": "scikit-learn", "version": sklearn.__version__},
            {"software": "LightGBM", "version": lightgbm.__version__},
            {"software": "matplotlib", "version": matplotlib_package.__version__},
        ]
    )

## 2. Data loading, validation, and sample construction

In [3]:
def load_and_clean_data(paths: ProjectPaths) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the modified Hansen data and document every cleaning restriction."""

    # Stop immediately with a useful error when the expected course dataset is absent.
    if not paths.data_file.exists():
        raise FileNotFoundError(f"Required dataset not found: {paths.data_file}")

    # Read the course-provided CSV into memory without changing the source file.
    raw_data = pd.read_csv(paths.data_file)

    # Record the raw row count before any type conversion or sample restriction.
    audit_rows: list[dict[str, Any]] = [
        {"step": "Raw dataset", "rows_remaining": len(raw_data), "rows_removed": 0}
    ]

    # Standardize column names so later code is insensitive to capitalization and extra spaces.
    data = raw_data.copy()
    data.columns = data.columns.str.lower().str.strip()

    # Define every variable required for the replication and causal extension.
    required_columns = {
        "low_score",
        "alcohol1",
        "alcohol2",
        "male",
        "white",
        "recidivism",
        "acc",
        "aged",
        "year",
    }

    # Identify missing columns before attempting any estimation.
    missing_columns = sorted(required_columns.difference(data.columns))
    if missing_columns:
        raise KeyError(f"The dataset is missing required columns: {missing_columns}")

    # Convert all required analysis variables to numeric values and turn invalid strings into missing values.
    for column in sorted(required_columns):
        data[column] = pd.to_numeric(data[column], errors="coerce")

    # Count and remove observations that cannot be used because a required variable is missing.
    rows_before_missing_filter = len(data)
    data = data.dropna(subset=sorted(required_columns)).copy()
    audit_rows.append(
        {
            "step": "Drop missing required variables",
            "rows_remaining": len(data),
            "rows_removed": rows_before_missing_filter - len(data),
        }
    )

    # Construct BAC from the integer low-score field to avoid floating-point threshold ambiguity.
    data["bac"] = data["low_score"] / 1000.0

    # Construct the minimum of the two raw alcohol readings as a validation check on low_score.
    data["calculated_low_score"] = data[["alcohol1", "alcohol2"]].min(axis=1)

    # Record whether the supplied low_score variable agrees with the two underlying readings.
    data["low_score_mismatch"] = (
        data["low_score"].astype(float) != data["calculated_low_score"].astype(float)
    ).astype(int)

    # Restrict the sample to the 1999-2007 period used by the main analysis.
    rows_before_year_filter = len(data)
    data = data[data["year"].between(1999, 2007, inclusive="both")].copy()
    audit_rows.append(
        {
            "step": "Restrict years to 1999-2007",
            "rows_remaining": len(data),
            "rows_removed": rows_before_year_filter - len(data),
        }
    )

    # Restrict the sample to drivers aged 21 or older because younger drivers face different legal thresholds.
    rows_before_age_filter = len(data)
    data = data[data["aged"] >= 21].copy()
    audit_rows.append(
        {
            "step": "Restrict age to 21 or older",
            "rows_remaining": len(data),
            "rows_removed": rows_before_age_filter - len(data),
        }
    )

    # Create exact 0.001 BAC clusters from the integer running variable.
    data["bac_bin"] = data["low_score"].astype(int) / 1000.0

    # Reconstruct the statutory treatment indicators from BAC rather than trusting supplied labels.
    data["dui_constructed"] = (data["bac"] >= 0.08).astype(int)
    data["agg_dui_constructed"] = (data["bac"] >= 0.15).astype(int)

    # Validate the binary outcome and predetermined indicator variables.
    binary_columns = ("male", "white", "recidivism", "acc")
    for column in binary_columns:
        invalid_values = sorted(set(data[column].unique()).difference({0, 1}))
        if invalid_values:
            raise ValueError(f"Column {column} contains non-binary values: {invalid_values}")

    # Sort by BAC and date order where possible so exported diagnostics are stable across runs.
    data = data.sort_values(["bac", "year"], kind="mergesort").reset_index(drop=True)

    # Convert the audit list into a table that documents the complete sample construction.
    audit_table = pd.DataFrame(audit_rows)

    # Return both the cleaned analysis sample and its cleaning audit.
    return data, audit_table


def treatment_validation_table(data: pd.DataFrame) -> pd.DataFrame:
    """Validate the running variable, analysis treatment rules, and supplied category indicators."""

    # Store each validation check with an interpretation of what equality should mean.
    validation_rows: list[dict[str, Any]] = []

    # Confirm that low_score is exactly the lower of the two raw BAC readings.
    validation_rows.append(
        {
            "check": "Running-variable construction",
            "comparison": "low_score equals min(Alcohol1, Alcohol2)",
            "mismatch_count": int(data["low_score_mismatch"].sum()),
            "interpretation": "Must equal zero for the selected running variable to be internally consistent.",
        }
    )

    # Confirm that the analysis treatment at 0.08 follows the stated sharp-RD rule by construction.
    analysis_dui_rule = (data["bac"] >= 0.08).astype(int)
    validation_rows.append(
        {
            "check": "Analysis treatment rule at 0.08",
            "comparison": "dui_constructed equals 1(low_score / 1000 >= 0.08)",
            "mismatch_count": int((data["dui_constructed"] != analysis_dui_rule).sum()),
            "interpretation": "Must equal zero; this is the treatment used in the sharp RD.",
        }
    )

    # Confirm that the analysis treatment at 0.15 follows the stated sharp-RD rule by construction.
    analysis_agg_rule = (data["bac"] >= 0.15).astype(int)
    validation_rows.append(
        {
            "check": "Analysis treatment rule at 0.15",
            "comparison": "agg_dui_constructed equals 1(low_score / 1000 >= 0.15)",
            "mismatch_count": int((data["agg_dui_constructed"] != analysis_agg_rule).sum()),
            "interpretation": "Must equal zero; this is the treatment used in the sharp RD.",
        }
    )

    # Document the coding of the supplied standard-DUI category when the variable is present.
    if "dui" in data.columns:
        supplied_dui = pd.to_numeric(data["dui"], errors="coerce")
        supplied_dui_rule = ((data["alcohol1"] > 80) & (data["alcohol1"] < 150)).astype(int)
        validation_rows.append(
            {
                "check": "Supplied standard-DUI category",
                "comparison": "dui equals 1(Alcohol1 > 80 and Alcohol1 < 150)",
                "mismatch_count": int((supplied_dui != supplied_dui_rule).sum()),
                "interpretation": "This supplied field is a mutually exclusive first-reading offense category, not the analysis threshold indicator based on low_score.",
            }
        )

    # Document the coding of the supplied aggravated-DUI category when the variable is present.
    if "agg_dui" in data.columns:
        supplied_agg_dui = pd.to_numeric(data["agg_dui"], errors="coerce")
        supplied_agg_rule = (data["alcohol1"] >= 150).astype(int)
        validation_rows.append(
            {
                "check": "Supplied aggravated-DUI category",
                "comparison": "agg_dui equals 1(Alcohol1 >= 150)",
                "mismatch_count": int((supplied_agg_dui != supplied_agg_rule).sum()),
                "interpretation": "This supplied field is based on the first BAC reading, not the low_score running variable used in the replication.",
            }
        )

    # Return a transparent audit that separates analysis treatment rules from supplied category fields.
    return pd.DataFrame(validation_rows)


def descriptive_statistics(data: pd.DataFrame) -> pd.DataFrame:
    """Summarize the variables used in the replication and causal extension."""

    # Select only variables that enter the empirical analysis.
    variables = ["bac", "recidivism", "aged", "male", "white", "acc", "year"]

    # Compute standard descriptive statistics and transpose them to one row per variable.
    summary = data[variables].describe().T.reset_index().rename(columns={"index": "variable"})

    # Return a rounded table suitable for CSV export.
    return summary.round(6)

## 3. Transparent local-polynomial RD estimators

In [4]:
def make_local_sample(data: pd.DataFrame, cutoff: float, bandwidth: float) -> pd.DataFrame:
    """Create one symmetric local sample and construct its RD regressors."""

    # Keep observations no farther than the requested bandwidth from the legal cutoff.
    local = data[data["bac"].between(cutoff - bandwidth, cutoff + bandwidth)].copy()

    # Define treatment using the exact statutory rule at the selected cutoff.
    local["treatment"] = (local["bac"] >= cutoff).astype(int)

    # Center BAC so the treatment coefficient is the discontinuity exactly at the cutoff.
    local["bac_centered"] = local["bac"] - cutoff

    # Allow the BAC slope to differ on the two sides of the legal threshold.
    local["treatment_x_bac"] = local["treatment"] * local["bac_centered"]

    # Create second-order terms for the quadratic robustness model only.
    local["bac_centered_sq"] = local["bac_centered"] ** 2
    local["treatment_x_bac_sq"] = local["treatment"] * local["bac_centered_sq"]

    # Return a fresh index so cross-fitting arrays align exactly with the local sample.
    return local.reset_index(drop=True)


def make_year_dummies(local: pd.DataFrame) -> pd.DataFrame:
    """Create a stable set of year fixed-effect indicators."""

    # Convert year to integer strings so dummy-column names are reproducible.
    year_series = local["year"].astype(int).astype(str)

    # Drop one year category to avoid exact collinearity with the intercept.
    return pd.get_dummies(year_series, prefix="year", drop_first=True, dtype=float)


def build_rd_design(
    local: pd.DataFrame,
    include_controls: bool,
    polynomial_order: int,
) -> pd.DataFrame:
    """Build the local-linear or local-quadratic RD design matrix."""

    # Start with the treatment jump and separate first-order BAC slopes on each side.
    design_columns = ["treatment", "bac_centered", "treatment_x_bac"]

    # Add second-order slope terms only for the explicitly labelled quadratic robustness model.
    if polynomial_order == 2:
        design_columns.extend(["bac_centered_sq", "treatment_x_bac_sq"])
    elif polynomial_order != 1:
        raise ValueError("polynomial_order must equal 1 or 2")

    # Copy the core RD regressors into a numeric design matrix.
    design = local[design_columns].astype(float).copy()

    # Add only predetermined covariates and year fixed effects in the adjusted specification.
    if include_controls:
        predetermined = local[list(PREDETERMINED_COVARIATES)].astype(float)
        design = pd.concat([design, predetermined, make_year_dummies(local)], axis=1)

    # Add an intercept so the treatment coefficient identifies the fitted jump at the cutoff.
    return sm.add_constant(design, has_constant="add")


def triangular_weights(local: pd.DataFrame, bandwidth: float) -> pd.Series:
    """Calculate triangular-kernel weights that decline linearly with distance from the cutoff."""

    # Assign the largest weight at the cutoff and zero weight at the edge of the local window.
    weights = 1.0 - local["bac_centered"].abs() / bandwidth

    # Clip small floating-point negatives to zero before weighted least squares estimation.
    return weights.clip(lower=0.0)


def fit_rd_model(
    data: pd.DataFrame,
    outcome: str,
    cutoff: float,
    bandwidth: float = BASELINE_BANDWIDTH,
    kernel: str = "rectangular",
    include_controls: bool = False,
    polynomial_order: int = 1,
) -> tuple[Any, pd.DataFrame, dict[str, Any]]:
    """Estimate one RD model with standard errors clustered by 0.001 BAC bins."""

    # Build the symmetric local sample and its centered running-variable terms.
    local = make_local_sample(data, cutoff=cutoff, bandwidth=bandwidth)

    # Verify that both sides of the threshold are represented before estimating the discontinuity.
    if local["treatment"].nunique() != 2:
        raise ValueError(f"Bandwidth {bandwidth} around {cutoff} does not contain both treatment states")

    # Verify that the requested outcome is available in the local sample.
    if outcome not in local.columns:
        raise KeyError(f"Outcome variable not found: {outcome}")

    # Construct the transparent local-polynomial regression design.
    design = build_rd_design(
        local,
        include_controls=include_controls,
        polynomial_order=polynomial_order,
    )

    # Convert the outcome to numeric form and stop if missing values remain.
    outcome_values = pd.to_numeric(local[outcome], errors="coerce").astype(float)
    if outcome_values.isna().any():
        raise ValueError(f"Outcome {outcome} contains missing values inside the estimation sample")

    # Estimate either an unweighted rectangular-kernel model or a triangular-kernel weighted model.
    if kernel == "rectangular":
        fitted_model = sm.OLS(outcome_values, design).fit(
            cov_type="cluster",
            cov_kwds={"groups": local["bac_bin"], "use_correction": True},
        )
    elif kernel == "triangular":
        fitted_model = sm.WLS(
            outcome_values,
            design,
            weights=triangular_weights(local, bandwidth),
        ).fit(
            cov_type="cluster",
            cov_kwds={"groups": local["bac_bin"], "use_correction": True},
        )
    else:
        raise ValueError("kernel must equal 'rectangular' or 'triangular'")

    # Extract the treatment discontinuity and its cluster-robust uncertainty measures.
    coefficient = float(fitted_model.params["treatment"])
    standard_error = float(fitted_model.bse["treatment"])
    confidence_interval = fitted_model.conf_int().loc["treatment"]

    # Store sample composition and the key estimate in a reusable result dictionary.
    result = {
        "cutoff": cutoff,
        "outcome": outcome,
        "bandwidth": bandwidth,
        "kernel": kernel,
        "polynomial_order": polynomial_order,
        "controls": include_controls,
        "estimate": coefficient,
        "standard_error": standard_error,
        "ci_lower": float(confidence_interval.iloc[0]),
        "ci_upper": float(confidence_interval.iloc[1]),
        "p_value": float(fitted_model.pvalues["treatment"]),
        "observations": int(fitted_model.nobs),
        "observations_below": int((local["bac"] < cutoff).sum()),
        "observations_at_or_above": int((local["bac"] >= cutoff).sum()),
        "clusters": int(local["bac_bin"].nunique()),
        "mean_below": float(local.loc[local["bac"] < cutoff, outcome].mean()),
        "mean_at_or_above": float(local.loc[local["bac"] >= cutoff, outcome].mean()),
    }

    # Return the fitted model, local data, and compact result record.
    return fitted_model, local, result

## 4. Preserve the reduced Table 2 balance replication

In [5]:
def replicate_table2_balance_tests(data: pd.DataFrame) -> pd.DataFrame:
    """Replicate the four available predetermined-characteristic balance tests."""

    # Map publication-style labels to the variable names in the modified course dataset.
    outcomes = {
        "Male": "male",
        "White": "white",
        "Age": "aged",
        "Accident": "acc",
    }

    # Create an empty list for one combined row per predetermined characteristic.
    table_rows: list[dict[str, Any]] = []

    # Estimate the same no-controls local-linear model at both legal thresholds.
    for label, variable in outcomes.items():
        _, _, dui_result = fit_rd_model(
            data,
            outcome=variable,
            cutoff=0.08,
            bandwidth=BASELINE_BANDWIDTH,
            kernel="rectangular",
            include_controls=False,
            polynomial_order=1,
        )
        _, _, aggravated_result = fit_rd_model(
            data,
            outcome=variable,
            cutoff=0.15,
            bandwidth=BASELINE_BANDWIDTH,
            kernel="rectangular",
            include_controls=False,
            polynomial_order=1,
        )

        # Reproduce the existing table structure so old and new outputs can be compared directly.
        table_rows.append(
            {
                "Characteristic": label,
                "DUI Coef.": dui_result["estimate"],
                "DUI SE": dui_result["standard_error"],
                "DUI Mean Below Cutoff": dui_result["mean_below"],
                "DUI Observations": dui_result["observations"],
                "Agg. DUI Coef.": aggravated_result["estimate"],
                "Agg. DUI SE": aggravated_result["standard_error"],
                "Agg. DUI Mean Below Cutoff": aggravated_result["mean_below"],
                "Agg. DUI Observations": aggravated_result["observations"],
            }
        )

    # Return estimates rounded exactly as in the original reduced replication output.
    return pd.DataFrame(table_rows).round(4)


def compare_balance_output(
    new_table: pd.DataFrame,
    existing_table_path: Path,
) -> pd.DataFrame:
    """Compare the extended code's balance estimates with the repository's saved table."""

    # Return a clear message when the historical comparison file is absent.
    if not existing_table_path.exists():
        return pd.DataFrame(
            [{"comparison": "Existing reduced table was not found", "maximum_absolute_difference": np.nan}]
        )

    # Load the repository's original reduced balance table.
    existing_table = pd.read_csv(existing_table_path)

    # Align rows by characteristic so file ordering cannot affect the comparison.
    merged = new_table.merge(existing_table, on="Characteristic", suffixes=("_new", "_existing"))

    # Identify numerical columns that are present in both versions of the table.
    comparable_columns = [
        column
        for column in existing_table.columns
        if column != "Characteristic" and f"{column}_new" in merged.columns
    ]

    # Calculate one maximum absolute difference for every comparable output column.
    comparison_rows = []
    for column in comparable_columns:
        difference = (
            pd.to_numeric(merged[f"{column}_new"], errors="coerce")
            - pd.to_numeric(merged[f"{column}_existing"], errors="coerce")
        ).abs()
        comparison_rows.append(
            {
                "comparison": column,
                "maximum_absolute_difference": float(difference.max()),
            }
        )

    # Return a compact audit showing whether the preserved replication changed.
    return pd.DataFrame(comparison_rows)

## 5. Recidivism RD replication and robustness checks

In [6]:
def estimate_recidivism_specifications(data: pd.DataFrame) -> pd.DataFrame:
    """Estimate baseline, adjusted, kernel, and polynomial recidivism RD models."""

    # Define transparent benchmark and robustness specifications before examining their results.
    specifications = [
        {
            "specification": "Baseline local linear RD",
            "bandwidth": 0.05,
            "kernel": "rectangular",
            "include_controls": False,
            "polynomial_order": 1,
        },
        {
            "specification": "Covariate-adjusted local linear RD",
            "bandwidth": 0.05,
            "kernel": "rectangular",
            "include_controls": True,
            "polynomial_order": 1,
        },
        {
            "specification": "Triangular-kernel adjusted RD",
            "bandwidth": 0.05,
            "kernel": "triangular",
            "include_controls": True,
            "polynomial_order": 1,
        },
        {
            "specification": "Quadratic adjusted robustness RD",
            "bandwidth": 0.05,
            "kernel": "rectangular",
            "include_controls": True,
            "polynomial_order": 2,
        },
    ]

    # Store one result row for every threshold-specification combination.
    result_rows: list[dict[str, Any]] = []

    # Estimate each pre-specified model separately at the standard and aggravated thresholds.
    for threshold_name, cutoff in THRESHOLDS.items():
        for specification in specifications:
            _, _, result = fit_rd_model(
                data,
                outcome="recidivism",
                cutoff=cutoff,
                bandwidth=specification["bandwidth"],
                kernel=specification["kernel"],
                include_controls=specification["include_controls"],
                polynomial_order=specification["polynomial_order"],
            )
            result["threshold_name"] = threshold_name
            result["specification"] = specification["specification"]
            result_rows.append(result)

    # Return all models in one tidy table for direct comparison.
    return pd.DataFrame(result_rows)


def estimate_bandwidth_sensitivity(data: pd.DataFrame) -> pd.DataFrame:
    """Estimate local-linear recidivism effects across bandwidths and kernels."""

    # Store one result row for each threshold, bandwidth, and kernel combination.
    sensitivity_rows: list[dict[str, Any]] = []

    # Re-estimate the no-controls local-linear benchmark over the pre-specified sensitivity grid.
    for threshold_name, cutoff in THRESHOLDS.items():
        for bandwidth in SENSITIVITY_BANDWIDTHS:
            for kernel in ("rectangular", "triangular"):
                _, _, result = fit_rd_model(
                    data,
                    outcome="recidivism",
                    cutoff=cutoff,
                    bandwidth=bandwidth,
                    kernel=kernel,
                    include_controls=False,
                    polynomial_order=1,
                )
                result["threshold_name"] = threshold_name
                sensitivity_rows.append(result)

    # Return the complete sensitivity grid as a tidy DataFrame.
    return pd.DataFrame(sensitivity_rows)


def rd_sample_diagnostics(data: pd.DataFrame) -> pd.DataFrame:
    """Report sample counts and raw recidivism rates on each side of each cutoff."""

    # Create one diagnostic record for each legal threshold using the baseline bandwidth.
    diagnostic_rows = []
    for threshold_name, cutoff in THRESHOLDS.items():
        local = make_local_sample(data, cutoff=cutoff, bandwidth=BASELINE_BANDWIDTH)
        diagnostic_rows.append(
            {
                "threshold_name": threshold_name,
                "cutoff": cutoff,
                "bandwidth": BASELINE_BANDWIDTH,
                "observations": len(local),
                "observations_below": int((local["treatment"] == 0).sum()),
                "observations_at_or_above": int((local["treatment"] == 1).sum()),
                "distinct_bac_bins": int(local["bac_bin"].nunique()),
                "recidivism_rate_below": float(local.loc[local["treatment"] == 0, "recidivism"].mean()),
                "recidivism_rate_at_or_above": float(
                    local.loc[local["treatment"] == 1, "recidivism"].mean()
                ),
            }
        )

    # Return the side-specific counts and rates used to contextualize the fitted discontinuity.
    return pd.DataFrame(diagnostic_rows)


def exploratory_density_diagnostic(data: pd.DataFrame) -> tuple[pd.DataFrame, dict[float, pd.DataFrame]]:
    """Estimate an exploratory discontinuity in log BAC-bin counts near each threshold."""

    # Store coefficient-level diagnostics and the binned density data used for plotting.
    diagnostic_rows: list[dict[str, Any]] = []
    binned_data: dict[float, pd.DataFrame] = {}

    # Analyze the count of observations at each discrete 0.001 BAC value near each threshold.
    for threshold_name, cutoff in THRESHOLDS.items():
        local = make_local_sample(data, cutoff=cutoff, bandwidth=BASELINE_BANDWIDTH)
        bins = (
            local.groupby("bac_bin", as_index=False)
            .size()
            .rename(columns={"size": "count"})
            .sort_values("bac_bin")
            .reset_index(drop=True)
        )

        # Construct the same separate-slope design using log bin counts as an exploratory density outcome.
        bins["treatment"] = (bins["bac_bin"] >= cutoff).astype(int)
        bins["bac_centered"] = bins["bac_bin"] - cutoff
        bins["treatment_x_bac"] = bins["treatment"] * bins["bac_centered"]
        bins["log_count"] = np.log(bins["count"])

        # Estimate heteroskedasticity-robust rather than clustered errors because the bin is the observation.
        density_design = sm.add_constant(
            bins[["treatment", "bac_centered", "treatment_x_bac"]].astype(float),
            has_constant="add",
        )
        density_model = sm.OLS(bins["log_count"], density_design).fit(cov_type="HC1")
        density_interval = density_model.conf_int().loc["treatment"]

        # Record the fitted log-count jump without calling it a formal McCrary test.
        diagnostic_rows.append(
            {
                "threshold_name": threshold_name,
                "cutoff": cutoff,
                "bandwidth": BASELINE_BANDWIDTH,
                "diagnostic": "Exploratory local-linear log bin-count discontinuity",
                "estimate": float(density_model.params["treatment"]),
                "standard_error": float(density_model.bse["treatment"]),
                "ci_lower": float(density_interval.iloc[0]),
                "ci_upper": float(density_interval.iloc[1]),
                "p_value": float(density_model.pvalues["treatment"]),
                "distinct_bins": len(bins),
            }
        )
        binned_data[cutoff] = bins

    # Return both the coefficient table and the raw bin counts used for visual inspection.
    return pd.DataFrame(diagnostic_rows), binned_data

## 6. RD diagnostic and robustness figures

In [7]:
def plot_recidivism_rd(data: pd.DataFrame, cutoff: float, figure_path: Path) -> None:
    """Plot binned recidivism rates and fitted local-linear RD lines near one cutoff."""

    # Estimate the baseline local-linear model and retain its local sample.
    model, local, _ = fit_rd_model(
        data,
        outcome="recidivism",
        cutoff=cutoff,
        bandwidth=BASELINE_BANDWIDTH,
        kernel="rectangular",
        include_controls=False,
        polynomial_order=1,
    )

    # Collapse individual outcomes into mean recidivism rates at each discrete BAC value.
    binned = (
        local.groupby("bac_bin", as_index=False)
        .agg(recidivism_rate=("recidivism", "mean"), observations=("recidivism", "size"))
        .sort_values("bac_bin")
    )

    # Create prediction grids separately below and above the cutoff to display the fitted discontinuity.
    below_grid = np.linspace(cutoff - BASELINE_BANDWIDTH, cutoff - 0.001, 100)
    above_grid = np.linspace(cutoff, cutoff + BASELINE_BANDWIDTH, 100)

    # Build the baseline RD design for an arbitrary BAC prediction grid.
    def prediction_design(bac_values: np.ndarray) -> pd.DataFrame:
        treatment_values = (bac_values >= cutoff).astype(int)
        centered_values = bac_values - cutoff
        return pd.DataFrame(
            {
                "const": np.ones(len(bac_values)),
                "treatment": treatment_values,
                "bac_centered": centered_values,
                "treatment_x_bac": treatment_values * centered_values,
            }
        )

    # Calculate fitted means and confidence intervals using the model's cluster-robust covariance matrix.
    below_prediction = model.get_prediction(prediction_design(below_grid)).summary_frame(alpha=0.05)
    above_prediction = model.get_prediction(prediction_design(above_grid)).summary_frame(alpha=0.05)

    # Create one standalone figure for this threshold.
    figure, axis = plt.subplots(figsize=(9, 6))

    # Plot observed mean recidivism at each BAC value with marker size reflecting bin sample size.
    marker_sizes = 15.0 + 85.0 * binned["observations"] / binned["observations"].max()
    axis.scatter(binned["bac_bin"], binned["recidivism_rate"], s=marker_sizes, alpha=0.55)

    # Plot the fitted left- and right-side local-linear response functions.
    axis.plot(below_grid, below_prediction["mean"], linewidth=2)
    axis.plot(above_grid, above_prediction["mean"], linewidth=2)

    # Add 95 percent confidence bands around each fitted side.
    axis.fill_between(
        below_grid,
        below_prediction["mean_ci_lower"],
        below_prediction["mean_ci_upper"],
        alpha=0.18,
    )
    axis.fill_between(
        above_grid,
        above_prediction["mean_ci_lower"],
        above_prediction["mean_ci_upper"],
        alpha=0.18,
    )

    # Mark the statutory threshold where punishment changes discontinuously.
    axis.axvline(cutoff, linestyle="--", linewidth=1.5)

    # Label the graph so it can be interpreted without referring back to the notebook.
    axis.set_title(f"Four-Year DUI Recidivism near BAC = {cutoff:.2f}")
    axis.set_xlabel("Blood alcohol content (BAC)")
    axis.set_ylabel("Mean four-year DUI recidivism")
    axis.grid(alpha=0.2)

    # Prevent labels from being clipped and save the publication-readable PNG.
    figure.tight_layout()
    figure.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.close(figure)


def plot_bandwidth_sensitivity(
    sensitivity_table: pd.DataFrame,
    cutoff: float,
    figure_path: Path,
) -> None:
    """Plot RD estimates and 95 percent intervals across bandwidths and kernels."""

    # Keep only the requested legal threshold and sort the estimates for stable plotting.
    subset = sensitivity_table[sensitivity_table["cutoff"] == cutoff].copy()
    subset = subset.sort_values(["kernel", "bandwidth"])

    # Create one figure for the requested threshold.
    figure, axis = plt.subplots(figsize=(9, 6))

    # Draw a separate line for each kernel so sensitivity can be assessed visually.
    for kernel, kernel_data in subset.groupby("kernel"):
        axis.errorbar(
            kernel_data["bandwidth"],
            kernel_data["estimate"],
            yerr=1.96 * kernel_data["standard_error"],
            marker="o",
            capsize=4,
            label=kernel.capitalize(),
        )

    # Mark a zero causal effect as the visual reference point.
    axis.axhline(0.0, linestyle="--", linewidth=1.2)

    # Add clear labels and a legend identifying the kernel specifications.
    axis.set_title(f"Bandwidth Sensitivity of the RD Estimate at BAC = {cutoff:.2f}")
    axis.set_xlabel("Symmetric bandwidth")
    axis.set_ylabel("Estimated change in recidivism probability")
    axis.legend(title="Kernel")
    axis.grid(alpha=0.2)

    # Save the standalone figure and release its memory.
    figure.tight_layout()
    figure.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.close(figure)


def plot_density_bins(bins: pd.DataFrame, cutoff: float, figure_path: Path) -> None:
    """Plot the number of observations at each discrete BAC value near one cutoff."""

    # Create one standalone figure for the BAC-bin density diagnostic.
    figure, axis = plt.subplots(figsize=(9, 6))

    # Plot observed counts at every 0.001 BAC bin in the baseline local window.
    axis.scatter(bins["bac_bin"], bins["count"], s=35, alpha=0.65)

    # Mark the legal threshold to make any visible bunching easy to assess.
    axis.axvline(cutoff, linestyle="--", linewidth=1.5)

    # Label the diagnostic as exploratory rather than a formal manipulation test.
    axis.set_title(f"Exploratory BAC-Bin Density Diagnostic near BAC = {cutoff:.2f}")
    axis.set_xlabel("Blood alcohol content (BAC)")
    axis.set_ylabel("Number of observations in 0.001 BAC bin")
    axis.grid(alpha=0.2)

    # Save the figure without displaying it interactively.
    figure.tight_layout()
    figure.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.close(figure)

## 7. Primary causal ML method: cross-fitted boosted-tree residualization inside RD

A standard causal forest is not used as the primary estimator because treatment in a sharp RD
is deterministically assigned by BAC. Instead, boosted trees predict recidivism using only
predetermined covariates and year. Out-of-fold residuals are then passed to the same local-linear
RD estimator. This can improve covariate adjustment while leaving identification at the BAC cutoff.

In [8]:
def ml_feature_matrix(local: pd.DataFrame, include_running_variable: bool) -> pd.DataFrame:
    """Build a stable ML feature matrix from predetermined variables and optional local BAC distance."""

    # Start with driver characteristics measured before the punishment triggered by the cutoff.
    features = local[list(PREDETERMINED_COVARIATES)].astype(float).copy()

    # Add a full set of year indicators because year is predetermined at the traffic stop.
    year_indicators = pd.get_dummies(
        local["year"].astype(int).astype(str),
        prefix="year",
        drop_first=False,
        dtype=float,
    )
    features = pd.concat([features, year_indicators], axis=1)

    # Add centered BAC only when estimating separate left- and right-side outcome response surfaces.
    if include_running_variable:
        features["bac_centered"] = local["bac_centered"].astype(float)

    # Return numeric features in a deterministic column order.
    return features.reindex(sorted(features.columns), axis=1)


def new_lgbm_classifier(seed: int) -> LGBMClassifier:
    """Create one shallow, regularized LightGBM probability model with a fixed seed."""

    # Combine the pre-specified model settings with the fold-specific random seed.
    parameters = {**LIGHTGBM_PARAMETERS, "random_state": seed}

    # Return a binary classifier whose probabilities estimate conditional recidivism risk.
    return LGBMClassifier(**parameters)


def cross_fitted_covariate_predictions(
    local: pd.DataFrame,
    seed: int,
) -> tuple[np.ndarray, dict[str, float]]:
    """Produce group-aware out-of-fold recidivism predictions using only predetermined covariates."""

    # Exclude treatment and BAC so the nuisance model cannot learn or remove the RD discontinuity mechanically.
    features = ml_feature_matrix(local, include_running_variable=False)

    # Store the binary recidivism outcome as an integer array for LightGBM.
    outcome = local["recidivism"].astype(int).to_numpy()

    # Group folds by exact BAC value so observations sharing a running-variable bin stay together.
    groups = local["bac_bin"].to_numpy()

    # Reserve an array for honest out-of-fold predicted recidivism probabilities.
    predictions = np.full(len(local), np.nan, dtype=float)

    # Split entire BAC bins across folds to limit leakage from repeated discrete running-variable values.
    cross_fitter = GroupKFold(n_splits=ML_CROSS_FOLDS)
    for fold_number, (training_index, validation_index) in enumerate(
        cross_fitter.split(features, outcome, groups=groups)
    ):
        # Fit one shallow boosted-tree model without using the held-out BAC bins.
        nuisance_model = new_lgbm_classifier(seed + fold_number)
        nuisance_model.fit(features.iloc[training_index], outcome[training_index])

        # Generate held-out probabilities that can be used for honest outcome residualization.
        predictions[validation_index] = nuisance_model.predict_proba(
            features.iloc[validation_index]
        )[:, 1]

    # Stop if a cross-fitting programming error left any observation without a prediction.
    if np.isnan(predictions).any():
        raise RuntimeError("Cross-fitting failed to produce a prediction for every local observation")

    # Calculate predictive diagnostics only to describe the nuisance model, not as causal evidence.
    diagnostics = {
        "brier_score": float(brier_score_loss(outcome, predictions)),
        "auc": float(roc_auc_score(outcome, predictions)),
        "mean_predicted_probability": float(predictions.mean()),
    }

    # Return the out-of-fold probabilities and their descriptive diagnostics.
    return predictions, diagnostics


def estimate_ml_residualized_rd(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Estimate the primary cross-fitted ML-assisted RD at each BAC threshold."""

    # Store one causal estimate and one nuisance-model diagnostic record per threshold.
    estimate_rows: list[dict[str, Any]] = []
    diagnostic_rows: list[dict[str, Any]] = []

    # Apply the same cross-fitted residualization procedure separately around each cutoff.
    for threshold_name, cutoff in THRESHOLDS.items():
        local = make_local_sample(data, cutoff=cutoff, bandwidth=BASELINE_BANDWIDTH)

        # Predict recidivism from predetermined covariates using out-of-fold boosted trees.
        nuisance_predictions, nuisance_diagnostics = cross_fitted_covariate_predictions(
            local,
            seed=RANDOM_SEED + int(round(cutoff * 1000)),
        )

        # Residualize recidivism without using treatment or BAC in the nuisance model.
        local["ml_residual"] = local["recidivism"].astype(float) - nuisance_predictions

        # Estimate the discontinuity in the honest residualized outcome using the original RD design.
        design = build_rd_design(local, include_controls=False, polynomial_order=1)
        residual_model = sm.OLS(local["ml_residual"], design).fit(
            cov_type="cluster",
            cov_kwds={"groups": local["bac_bin"], "use_correction": True},
        )
        interval = residual_model.conf_int().loc["treatment"]

        # Record the causal discontinuity and the exact local sample used to estimate it.
        estimate_rows.append(
            {
                "threshold_name": threshold_name,
                "cutoff": cutoff,
                "method": "Cross-fitted boosted-tree residualized local-linear RD",
                "estimate": float(residual_model.params["treatment"]),
                "standard_error": float(residual_model.bse["treatment"]),
                "ci_lower": float(interval.iloc[0]),
                "ci_upper": float(interval.iloc[1]),
                "p_value": float(residual_model.pvalues["treatment"]),
                "observations": int(residual_model.nobs),
                "observations_below": int((local["treatment"] == 0).sum()),
                "observations_at_or_above": int((local["treatment"] == 1).sum()),
                "clusters": int(local["bac_bin"].nunique()),
            }
        )

        # Record prediction diagnostics separately so they are not confused with causal validation.
        diagnostic_rows.append(
            {
                "threshold_name": threshold_name,
                "cutoff": cutoff,
                "method": "Predetermined-covariate LightGBM nuisance model",
                **nuisance_diagnostics,
                "cross_folds": ML_CROSS_FOLDS,
            }
        )

    # Return the causal estimates and descriptive nuisance-model diagnostics as separate tables.
    return pd.DataFrame(estimate_rows), pd.DataFrame(diagnostic_rows)

## 8. Exploratory heterogeneous effects: local left/right boosted outcome models

The following T-learner estimates separate conditional recidivism response surfaces on the two
sides of each cutoff and evaluates both surfaces at the cutoff. It preserves the RD estimand as
a difference between conditional left and right limits. Because the modified dataset has few
covariates, its CATE results are explicitly exploratory rather than the primary causal evidence.

In [9]:
def cross_fitted_local_t_learner(
    local: pd.DataFrame,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Estimate honest individual-level conditional discontinuities at the cutoff."""

    # Include centered BAC to model each side's local response surface, but evaluate it at zero later.
    features = ml_feature_matrix(local, include_running_variable=True)

    # Create a cutoff-evaluation copy where every observation is evaluated exactly at centered BAC zero.
    cutoff_features = features.copy()
    cutoff_features["bac_centered"] = 0.0

    # Store outcome, treatment, and group labels as aligned arrays.
    outcome = local["recidivism"].astype(int).to_numpy()
    treatment = local["treatment"].astype(int).to_numpy()
    groups = local["bac_bin"].to_numpy()

    # Reserve arrays for out-of-fold left-limit and right-limit predictions.
    predicted_left_limit = np.full(len(local), np.nan, dtype=float)
    predicted_right_limit = np.full(len(local), np.nan, dtype=float)

    # Keep fold-level prediction diagnostics for both side-specific models.
    fold_diagnostics: list[dict[str, Any]] = []

    # Hold out complete BAC bins so the same discrete running-variable value never appears in both fold roles.
    cross_fitter = GroupKFold(n_splits=ML_CROSS_FOLDS)
    for fold_number, (training_index, validation_index) in enumerate(
        cross_fitter.split(features, outcome, groups=groups)
    ):
        # Identify left- and right-side observations inside the training folds.
        left_training_index = training_index[treatment[training_index] == 0]
        right_training_index = training_index[treatment[training_index] == 1]

        # Require adequate training observations on each side before estimating conditional limits.
        if len(left_training_index) < 500 or len(right_training_index) < 500:
            raise ValueError("A cross-fitting fold contains too few observations on one side of the cutoff")

        # Fit the left-side outcome model using only observations below the BAC cutoff.
        left_model = new_lgbm_classifier(seed + 10 * fold_number + 1)
        left_model.fit(features.iloc[left_training_index], outcome[left_training_index])

        # Fit the right-side outcome model using only observations at or above the BAC cutoff.
        right_model = new_lgbm_classifier(seed + 10 * fold_number + 2)
        right_model.fit(features.iloc[right_training_index], outcome[right_training_index])

        # Predict both conditional limits at BAC centered equal to zero for held-out drivers.
        predicted_left_limit[validation_index] = left_model.predict_proba(
            cutoff_features.iloc[validation_index]
        )[:, 1]
        predicted_right_limit[validation_index] = right_model.predict_proba(
            cutoff_features.iloc[validation_index]
        )[:, 1]

        # Store side-specific training sizes so honesty and overlap can be audited.
        fold_diagnostics.append(
            {
                "fold": fold_number + 1,
                "left_training_observations": len(left_training_index),
                "right_training_observations": len(right_training_index),
                "validation_observations": len(validation_index),
            }
        )

    # Stop if any held-out observation lacks one of the two conditional-limit predictions.
    if np.isnan(predicted_left_limit).any() or np.isnan(predicted_right_limit).any():
        raise RuntimeError("The local T-learner did not produce complete out-of-fold predictions")

    # Define each driver's exploratory conditional treatment effect as right limit minus left limit.
    cate = predicted_right_limit - predicted_left_limit

    # Attach the honest limit predictions and CATEs to the local analysis sample.
    cate_data = local.copy()
    cate_data["predicted_left_limit"] = predicted_left_limit
    cate_data["predicted_right_limit"] = predicted_right_limit
    cate_data["cate"] = cate

    # Return individual-level CATE data and cross-fitting sample diagnostics.
    return cate_data, pd.DataFrame(fold_diagnostics)


def age_group_series(age: pd.Series) -> pd.Series:
    """Create interpretable age groups for exploratory treatment-effect summaries."""

    # Divide age into broad groups that retain adequate sample sizes in each local window.
    return pd.cut(
        age,
        bins=[20, 29, 39, 49, np.inf],
        labels=["21-29", "30-39", "40-49", "50+"],
        right=True,
    )


def subgroup_masks(cate_data: pd.DataFrame) -> dict[str, pd.Series]:
    """Return named subgroup masks used for interpretable CATE summaries."""

    # Construct the age categories once so each bootstrap uses the same subgroup definitions.
    age_groups = age_group_series(cate_data["aged"])

    # Define subgroups only from predetermined characteristics available in the modified dataset.
    return {
        "Age: 21-29": age_groups == "21-29",
        "Age: 30-39": age_groups == "30-39",
        "Age: 40-49": age_groups == "40-49",
        "Age: 50+": age_groups == "50+",
        "Female": cate_data["male"] == 0,
        "Male": cate_data["male"] == 1,
        "Non-white": cate_data["white"] == 0,
        "White": cate_data["white"] == 1,
        "No accident": cate_data["acc"] == 0,
        "Accident": cate_data["acc"] == 1,
    }


def fit_weighted_side_models(
    local: pd.DataFrame,
    bootstrap_weights: pd.Series,
    seed: int,
) -> tuple[LGBMClassifier, LGBMClassifier, pd.DataFrame]:
    """Fit weighted left- and right-side boosted models for one BAC-bin bootstrap draw."""

    # Build the local outcome-model features including centered BAC.
    features = ml_feature_matrix(local, include_running_variable=True)

    # Extract the observed binary recidivism outcome.
    outcome = local["recidivism"].astype(int)

    # Identify left- and right-side observations with positive bootstrap multiplicity.
    left_mask = (local["treatment"] == 0) & (bootstrap_weights > 0)
    right_mask = (local["treatment"] == 1) & (bootstrap_weights > 0)

    # Fit the below-cutoff response surface using BAC-bin bootstrap weights.
    left_model = new_lgbm_classifier(seed + 1)
    left_model.fit(
        features.loc[left_mask],
        outcome.loc[left_mask],
        sample_weight=bootstrap_weights.loc[left_mask],
    )

    # Fit the at-or-above-cutoff response surface using BAC-bin bootstrap weights.
    right_model = new_lgbm_classifier(seed + 2)
    right_model.fit(
        features.loc[right_mask],
        outcome.loc[right_mask],
        sample_weight=bootstrap_weights.loc[right_mask],
    )

    # Evaluate both response surfaces at the BAC cutoff for the original covariate distribution.
    cutoff_features = features.copy()
    cutoff_features["bac_centered"] = 0.0

    # Return the fitted side models and common cutoff-evaluation feature matrix.
    return left_model, right_model, cutoff_features


def draw_bac_bin_bootstrap_weights(local: pd.DataFrame, rng: np.random.Generator) -> pd.Series:
    """Generate cluster-bootstrap multiplicities by resampling BAC bins separately on each side."""

    # Start every observation with zero weight until its BAC bin is sampled.
    weights = pd.Series(0.0, index=local.index)

    # Resample distinct bins separately below and above the cutoff to preserve both local sides.
    for treatment_value in (0, 1):
        side_bins = local.loc[local["treatment"] == treatment_value, "bac_bin"].unique()
        sampled_bins = rng.choice(side_bins, size=len(side_bins), replace=True)
        multiplicities = pd.Series(sampled_bins).value_counts()

        # Assign each observation the number of times its complete BAC bin was selected.
        side_mask = local["treatment"] == treatment_value
        weights.loc[side_mask] = (
            local.loc[side_mask, "bac_bin"].map(multiplicities).fillna(0.0).astype(float)
        )

    # Return observation weights that implement the discrete-running-variable cluster bootstrap.
    return weights


def bootstrap_cate_summaries(
    cate_data: pd.DataFrame,
    replications: int,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Bootstrap the average and subgroup local T-learner effects by BAC bin."""

    # Initialize a reproducible random-number generator for all bootstrap draws.
    rng = np.random.default_rng(seed)

    # Precompute subgroup masks on the fixed evaluation sample.
    masks = subgroup_masks(cate_data)

    # Store the average and subgroup effect from every bootstrap replication.
    average_draws: list[dict[str, Any]] = []
    subgroup_draws: list[dict[str, Any]] = []

    # Refit the two side-specific models after resampling complete BAC bins.
    for replication in range(replications):
        bootstrap_weights = draw_bac_bin_bootstrap_weights(cate_data, rng)
        left_model, right_model, cutoff_features = fit_weighted_side_models(
            cate_data,
            bootstrap_weights=bootstrap_weights,
            seed=seed + 1000 + 10 * replication,
        )

        # Calculate the right-minus-left conditional discontinuity for the fixed evaluation sample.
        left_predictions = left_model.predict_proba(cutoff_features)[:, 1]
        right_predictions = right_model.predict_proba(cutoff_features)[:, 1]
        bootstrap_cate = right_predictions - left_predictions

        # Save the bootstrap average effect for uncertainty around the exploratory ML estimand.
        average_draws.append(
            {
                "replication": replication + 1,
                "average_effect": float(np.mean(bootstrap_cate)),
            }
        )

        # Save subgroup means from the same bootstrap draw for interpretable heterogeneity intervals.
        for subgroup, mask in masks.items():
            mask_array = mask.to_numpy(dtype=bool)
            subgroup_draws.append(
                {
                    "replication": replication + 1,
                    "subgroup": subgroup,
                    "effect": float(np.mean(bootstrap_cate[mask_array])),
                }
            )

    # Return raw bootstrap draws so percentile intervals and standard errors remain transparent.
    return pd.DataFrame(average_draws), pd.DataFrame(subgroup_draws)


def summarize_cate_results(
    threshold_name: str,
    cutoff: float,
    cate_data: pd.DataFrame,
    average_draws: pd.DataFrame,
    subgroup_draws: pd.DataFrame,
) -> tuple[dict[str, Any], pd.DataFrame]:
    """Summarize the exploratory average and subgroup conditional discontinuities."""

    # Calculate the honest cross-fitted average CATE point estimate.
    average_effect = float(cate_data["cate"].mean())

    # Use the BAC-bin bootstrap distribution for an exploratory standard error and percentile interval.
    bootstrap_standard_error = float(average_draws["average_effect"].std(ddof=1))
    average_interval = average_draws["average_effect"].quantile([0.025, 0.975])

    # Create one compact average-effect record for the threshold.
    average_summary = {
        "threshold_name": threshold_name,
        "cutoff": cutoff,
        "method": "Exploratory cross-fitted local boosted T-learner",
        "estimate": average_effect,
        "standard_error": bootstrap_standard_error,
        "ci_lower": float(average_interval.loc[0.025]),
        "ci_upper": float(average_interval.loc[0.975]),
        "observations": len(cate_data),
        "observations_below": int((cate_data["treatment"] == 0).sum()),
        "observations_at_or_above": int((cate_data["treatment"] == 1).sum()),
        "bootstrap_replications": len(average_draws),
    }

    # Calculate point estimates for each pre-specified predetermined subgroup.
    masks = subgroup_masks(cate_data)
    subgroup_rows: list[dict[str, Any]] = []
    for subgroup, mask in masks.items():
        subgroup_bootstrap = subgroup_draws.loc[subgroup_draws["subgroup"] == subgroup, "effect"]
        subgroup_interval = subgroup_bootstrap.quantile([0.025, 0.975])
        subgroup_rows.append(
            {
                "threshold_name": threshold_name,
                "cutoff": cutoff,
                "subgroup": subgroup,
                "observations": int(mask.sum()),
                "estimate": float(cate_data.loc[mask, "cate"].mean()),
                "bootstrap_standard_error": float(subgroup_bootstrap.std(ddof=1)),
                "ci_lower": float(subgroup_interval.loc[0.025]),
                "ci_upper": float(subgroup_interval.loc[0.975]),
                "bootstrap_replications": int(subgroup_bootstrap.shape[0]),
            }
        )

    # Return both the average effect and all subgroup estimates.
    return average_summary, pd.DataFrame(subgroup_rows)


def seed_sensitivity_for_t_learner(
    local: pd.DataFrame,
    seeds: Iterable[int] = (610, 1610, 2610),
) -> pd.DataFrame:
    """Check whether the exploratory average CATE changes materially across random seeds."""

    # Re-estimate the honest local T-learner under each pre-specified seed.
    rows = []
    for seed in seeds:
        cate_data, _ = cross_fitted_local_t_learner(local, seed=seed)
        rows.append({"seed": seed, "average_cate": float(cate_data["cate"].mean())})

    # Return the seed-level average effects as a stability diagnostic.
    return pd.DataFrame(rows)


def final_side_model_feature_importance(
    local: pd.DataFrame,
    seed: int,
) -> pd.DataFrame:
    """Summarize predictive split-gain importance from final left and right outcome models."""

    # Give every observation unit weight so the helper can fit on the full local sample.
    unit_weights = pd.Series(1.0, index=local.index)

    # Fit one final model on each side for descriptive feature-importance reporting.
    left_model, right_model, cutoff_features = fit_weighted_side_models(
        local,
        bootstrap_weights=unit_weights,
        seed=seed,
    )

    # Extract gain-based importance, which is less sensitive to split count than raw feature frequency.
    left_gain = left_model.booster_.feature_importance(importance_type="gain")
    right_gain = right_model.booster_.feature_importance(importance_type="gain")

    # Average normalized gain across the two side models while guarding against zero total gain.
    left_normalized = left_gain / left_gain.sum() if left_gain.sum() > 0 else left_gain
    right_normalized = right_gain / right_gain.sum() if right_gain.sum() > 0 else right_gain
    average_normalized = (left_normalized + right_normalized) / 2.0

    # Return a sorted table explicitly labelled as predictive rather than causal importance.
    importance = pd.DataFrame(
        {
            "feature": cutoff_features.columns,
            "left_normalized_gain": left_normalized,
            "right_normalized_gain": right_normalized,
            "average_normalized_gain": average_normalized,
        }
    )
    return importance.sort_values("average_normalized_gain", ascending=False).reset_index(drop=True)


def estimate_exploratory_heterogeneity(
    data: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[float, pd.DataFrame], dict[float, pd.DataFrame]]:
    """Estimate exploratory local T-learner CATEs and their BAC-bin bootstrap uncertainty."""

    # Store average effects, subgroup effects, fold diagnostics, and random-seed sensitivity records.
    average_rows: list[dict[str, Any]] = []
    subgroup_tables: list[pd.DataFrame] = []
    fold_tables: list[pd.DataFrame] = []
    seed_tables: list[pd.DataFrame] = []

    # Retain individual CATE data and feature importance for threshold-specific figures.
    cate_data_by_cutoff: dict[float, pd.DataFrame] = {}
    importance_by_cutoff: dict[float, pd.DataFrame] = {}

    # Estimate the local left/right outcome models separately at each legal threshold.
    for threshold_name, cutoff in THRESHOLDS.items():
        local = make_local_sample(data, cutoff=cutoff, bandwidth=BASELINE_BANDWIDTH)
        cate_data, fold_diagnostics = cross_fitted_local_t_learner(
            local,
            seed=RANDOM_SEED + int(round(cutoff * 1000)),
        )

        # Generate BAC-bin cluster bootstrap draws for exploratory ML uncertainty intervals.
        average_draws, subgroup_draws = bootstrap_cate_summaries(
            cate_data,
            replications=ML_BOOTSTRAP_REPLICATIONS,
            seed=RANDOM_SEED + 5000 + int(round(cutoff * 1000)),
        )

        # Summarize the honest point estimates and bootstrap uncertainty.
        average_summary, subgroup_summary = summarize_cate_results(
            threshold_name,
            cutoff,
            cate_data,
            average_draws,
            subgroup_draws,
        )
        average_rows.append(average_summary)
        subgroup_tables.append(subgroup_summary)

        # Label fold diagnostics with their threshold before combining them.
        fold_diagnostics["threshold_name"] = threshold_name
        fold_diagnostics["cutoff"] = cutoff
        fold_tables.append(fold_diagnostics)

        # Re-estimate the exploratory average under multiple seeds to assess algorithmic stability.
        seed_table = seed_sensitivity_for_t_learner(local)
        seed_table["threshold_name"] = threshold_name
        seed_table["cutoff"] = cutoff
        seed_tables.append(seed_table)

        # Fit final side models solely for descriptive predictive feature-importance reporting.
        importance_by_cutoff[cutoff] = final_side_model_feature_importance(
            local,
            seed=RANDOM_SEED + 9000 + int(round(cutoff * 1000)),
        )
        cate_data_by_cutoff[cutoff] = cate_data

    # Concatenate threshold-level outputs into tidy project tables.
    return (
        pd.DataFrame(average_rows),
        pd.concat(subgroup_tables, ignore_index=True),
        pd.concat(fold_tables, ignore_index=True),
        pd.concat(seed_tables, ignore_index=True),
        cate_data_by_cutoff,
        importance_by_cutoff,
    )

## 9. Exploratory heterogeneity figures

In [10]:
def plot_cate_distribution(cate_data: pd.DataFrame, cutoff: float, figure_path: Path) -> None:
    """Plot the distribution of exploratory conditional discontinuity estimates."""

    # Create a standalone histogram for the threshold-specific CATE distribution.
    figure, axis = plt.subplots(figsize=(9, 6))
    axis.hist(cate_data["cate"], bins=40, alpha=0.75)

    # Mark the mean exploratory CATE as a reference line.
    axis.axvline(cate_data["cate"].mean(), linestyle="--", linewidth=1.5)

    # Label the plot carefully so model-generated heterogeneity is not overstated.
    axis.set_title(f"Exploratory Local T-Learner CATE Distribution at BAC = {cutoff:.2f}")
    axis.set_xlabel("Estimated conditional change in recidivism probability")
    axis.set_ylabel("Number of local observations")
    axis.grid(alpha=0.2)

    # Save the figure and close it to avoid accumulating open graphics objects.
    figure.tight_layout()
    figure.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.close(figure)


def plot_sorted_cates(cate_data: pd.DataFrame, cutoff: float, figure_path: Path) -> None:
    """Plot sorted exploratory CATEs to show their model-implied dispersion."""

    # Sort the individual CATE estimates from most negative to most positive.
    sorted_cates = np.sort(cate_data["cate"].to_numpy())
    percentile_rank = np.linspace(0.0, 100.0, len(sorted_cates))

    # Create a standalone sorted-effect plot for the selected threshold.
    figure, axis = plt.subplots(figsize=(9, 6))
    axis.plot(percentile_rank, sorted_cates, linewidth=2)
    axis.axhline(0.0, linestyle="--", linewidth=1.2)

    # Label the axes in terms of estimated recidivism probability changes.
    axis.set_title(f"Sorted Exploratory CATEs at BAC = {cutoff:.2f}")
    axis.set_xlabel("Percentile of local observations")
    axis.set_ylabel("Estimated conditional change in recidivism probability")
    axis.grid(alpha=0.2)

    # Save and close the figure.
    figure.tight_layout()
    figure.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.close(figure)


def plot_subgroup_effects(subgroups: pd.DataFrame, cutoff: float, figure_path: Path) -> None:
    """Plot exploratory subgroup CATE means and BAC-bin bootstrap intervals."""

    # Keep and reverse the threshold-specific subgroup order for readable horizontal plotting.
    subset = subgroups[subgroups["cutoff"] == cutoff].copy().iloc[::-1]

    # Create asymmetric error-bar widths from the percentile confidence bounds.
    lower_errors = subset["estimate"] - subset["ci_lower"]
    upper_errors = subset["ci_upper"] - subset["estimate"]

    # Create one standalone horizontal coefficient plot.
    figure, axis = plt.subplots(figsize=(10, 7))
    axis.errorbar(
        subset["estimate"],
        subset["subgroup"],
        xerr=np.vstack([lower_errors, upper_errors]),
        fmt="o",
        capsize=4,
    )
    axis.axvline(0.0, linestyle="--", linewidth=1.2)

    # Emphasize that these intervals summarize an exploratory ML specification.
    axis.set_title(f"Exploratory Subgroup CATEs at BAC = {cutoff:.2f}")
    axis.set_xlabel("Estimated conditional change in recidivism probability")
    axis.set_ylabel("Predetermined subgroup")
    axis.grid(alpha=0.2)

    # Save and close the figure.
    figure.tight_layout()
    figure.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.close(figure)


def plot_feature_importance(importance: pd.DataFrame, cutoff: float, figure_path: Path) -> None:
    """Plot descriptive predictive gain importance from the local side models."""

    # Keep the ten most influential prediction features to preserve readability.
    subset = importance.head(10).sort_values("average_normalized_gain", ascending=True)

    # Create one standalone horizontal importance plot.
    figure, axis = plt.subplots(figsize=(9, 6))
    axis.barh(subset["feature"], subset["average_normalized_gain"])

    # Label the plot as predictive importance rather than a causal ranking of moderators.
    axis.set_title(f"Predictive Feature Importance in Local Side Models at BAC = {cutoff:.2f}")
    axis.set_xlabel("Average normalized gain across left and right models")
    axis.set_ylabel("Feature")
    axis.grid(alpha=0.2)

    # Save and close the figure.
    figure.tight_layout()
    figure.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.close(figure)

## 10. Output export, comparison tables, and execution summary

In [11]:
def comparison_table(
    recidivism_results: pd.DataFrame,
    ml_rd_results: pd.DataFrame,
    exploratory_average_results: pd.DataFrame,
) -> pd.DataFrame:
    """Combine the main conventional and ML-assisted estimates in one comparison table."""

    # Keep the baseline and adjusted transparent RD benchmarks.
    conventional = recidivism_results[
        recidivism_results["specification"].isin(
            ["Baseline local linear RD", "Covariate-adjusted local linear RD"]
        )
    ].copy()
    conventional["method"] = conventional["specification"]

    # Select common estimate columns before stacking different estimator families.
    common_columns = [
        "threshold_name",
        "cutoff",
        "method",
        "estimate",
        "standard_error",
        "ci_lower",
        "ci_upper",
        "p_value",
        "observations",
        "observations_below",
        "observations_at_or_above",
    ]

    # Add p-values as missing for the exploratory percentile-bootstrap T-learner interval.
    exploratory = exploratory_average_results.copy()
    exploratory["p_value"] = np.nan

    # Stack all primary and exploratory causal estimates with aligned column names.
    return pd.concat(
        [
            conventional[common_columns],
            ml_rd_results[common_columns],
            exploratory[common_columns],
        ],
        ignore_index=True,
    )


def format_effect(value: float) -> str:
    """Format a probability effect as percentage points for the narrative summary."""

    # Convert a probability-unit estimate into percentage points with two decimals.
    return f"{100.0 * value:.2f} percentage points"


def write_execution_summary(
    paths: ProjectPaths,
    balance_comparison: pd.DataFrame,
    recidivism_results: pd.DataFrame,
    ml_rd_results: pd.DataFrame,
    exploratory_average_results: pd.DataFrame,
    density_results: pd.DataFrame,
) -> None:
    """Write a concise Markdown audit and results summary generated from the executed outputs."""

    # Extract baseline, primary ML-assisted, and exploratory estimates by cutoff.
    baseline = recidivism_results[
        recidivism_results["specification"] == "Baseline local linear RD"
    ].set_index("cutoff")
    ml_primary = ml_rd_results.set_index("cutoff")
    exploratory = exploratory_average_results.set_index("cutoff")
    density = density_results.set_index("cutoff")

    # Confirm whether every preserved balance output matches the repository's saved rounded table.
    maximum_balance_difference = balance_comparison["maximum_absolute_difference"].max()
    balance_statement = (
        "The preserved balance replication matches the repository's saved rounded table exactly."
        if pd.notna(maximum_balance_difference) and maximum_balance_difference == 0
        else f"The largest rounded balance-table difference is {maximum_balance_difference}."
    )

    # Build a readable Markdown report using only values produced by the executed code.
    lines = [
        "# Execution Summary",
        "",
        "## Audit",
        "",
        balance_statement,
        "The original notebook contained two Colab-specific code cells and only reproduced four Table 2 balance outcomes. ",
        "The extension adds recidivism RD estimates, bandwidth/kernel/polynomial checks, an exploratory density diagnostic, ",
        "cross-fitted ML covariate adjustment inside the RD, and exploratory heterogeneous conditional discontinuities.",
        "",
        "## Primary causal estimates",
        "",
    ]

    # Add threshold-specific baseline and ML-assisted estimates to the summary.
    for threshold_name, cutoff in THRESHOLDS.items():
        baseline_row = baseline.loc[cutoff]
        ml_row = ml_primary.loc[cutoff]
        exploratory_row = exploratory.loc[cutoff]
        density_row = density.loc[cutoff]
        lines.extend(
            [
                f"### {threshold_name} (BAC = {cutoff:.2f})",
                "",
                f"- Baseline local-linear RD: {format_effect(baseline_row['estimate'])} "
                f"(95% CI {format_effect(baseline_row['ci_lower'])} to {format_effect(baseline_row['ci_upper'])}).",
                f"- Cross-fitted boosted-tree residualized RD: {format_effect(ml_row['estimate'])} "
                f"(95% CI {format_effect(ml_row['ci_lower'])} to {format_effect(ml_row['ci_upper'])}).",
                f"- Exploratory local boosted T-learner average: {format_effect(exploratory_row['estimate'])} "
                f"(BAC-bin bootstrap interval {format_effect(exploratory_row['ci_lower'])} to "
                f"{format_effect(exploratory_row['ci_upper'])}).",
                f"- Exploratory log bin-count discontinuity p-value: {density_row['p_value']:.4f}.",
                "",
            ]
        )

    # Add the method-selection conclusion and central limitations.
    lines.extend(
        [
            "## Method selection",
            "",
            "The primary ML method is cross-fitted boosted-tree outcome residualization followed by the original local-linear RD. ",
            "It does not estimate a propensity score and does not treat the full dataset as unconfounded. Treatment remains identified ",
            "by the statutory BAC cutoff. A standard causal forest is not used as the primary estimator because treatment is a deterministic ",
            "function of BAC in this sharp RD.",
            "",
            "The local boosted T-learner is retained only as an exploratory heterogeneity exercise. It estimates separate conditional left and ",
            "right limits at the cutoff, but its subgroup differences should not be treated as strong evidence because the modified dataset has ",
            "few predetermined covariates and the CATE estimates are model-dependent.",
            "",
            "## Reproduction",
            "",
            "Run `python code/Replication_and_Causal_ML_Extension.py` from the repository root, or open and run the matching notebook from top to bottom.",
        ]
    )

    # Write the generated summary into the repository root without modifying the original report PDF.
    (paths.root / "EXECUTION_SUMMARY.md").write_text("\n".join(lines), encoding="utf-8")


def save_json_metadata(paths: ProjectPaths) -> None:
    """Save core reproducibility settings in a machine-readable JSON file."""

    # Record all stochastic and specification choices that affect the generated outputs.
    metadata = {
        "random_seed": RANDOM_SEED,
        "thresholds": THRESHOLDS,
        "baseline_bandwidth": BASELINE_BANDWIDTH,
        "sensitivity_bandwidths": list(SENSITIVITY_BANDWIDTHS),
        "predetermined_covariates": list(PREDETERMINED_COVARIATES),
        "ml_cross_folds": ML_CROSS_FOLDS,
        "ml_bootstrap_replications": ML_BOOTSTRAP_REPLICATIONS,
        "lightgbm_parameters": LIGHTGBM_PARAMETERS,
    }

    # Save indented JSON so settings can be inspected without opening the notebook.
    (paths.table_directory / "analysis_metadata.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

## 11. Execute the complete project

In [12]:
def main() -> None:
    """Run the complete replication, causal extension, diagnostics, and output export workflow."""

    # Offer a file upload only when running in Colab and the dataset is not already present.
    maybe_upload_dataset_in_colab()

    # Locate the repository and create all output folders before reading or estimating anything.
    project_root = locate_project_root()
    paths = build_project_paths(project_root)

    # Print the resolved root so local and Colab users can verify where outputs are being written.
    print(f"Project root: {paths.root}")

    # Record and export the exact software versions used in this execution.
    versions = environment_versions()
    versions.to_csv(paths.table_directory / "software_versions.csv", index=False)
    print(versions.to_string(index=False))

    # Load, validate, and restrict the dataset while documenting every removed observation.
    data, sample_audit = load_and_clean_data(paths)
    sample_audit.to_csv(paths.table_directory / "sample_construction_audit.csv", index=False)
    print("\nSample construction audit:")
    print(sample_audit.to_string(index=False))

    # Export treatment and running-variable consistency checks.
    treatment_audit = treatment_validation_table(data)
    treatment_audit.to_csv(paths.table_directory / "treatment_validation.csv", index=False)

    # Export descriptive statistics for all variables used in the extension.
    descriptives = descriptive_statistics(data)
    descriptives.to_csv(paths.table_directory / "descriptive_statistics.csv", index=False)

    # Identify the repository's historical balance output before writing the extended results.
    historical_balance_path = paths.table_directory / "reduced_table2_replication.csv"
    original_archive_path = paths.table_directory / "reduced_table2_replication_original.csv"

    # Preserve the historical file under a new name on the first extended run.
    if historical_balance_path.exists() and not original_archive_path.exists():
        historical_before_overwrite = pd.read_csv(historical_balance_path)
        historical_before_overwrite.to_csv(original_archive_path, index=False)

    # Preserve the original reduced Table 2 balance replication in both old and clearer filenames.
    balance_table = replicate_table2_balance_tests(data)
    balance_table.to_csv(paths.table_directory / "table2_balance_replication.csv", index=False)
    balance_table.to_csv(historical_balance_path, index=False)

    # Compare against the archived historical table so later reruns remain meaningful.
    comparison_source = original_archive_path if original_archive_path.exists() else historical_balance_path
    balance_comparison = compare_balance_output(balance_table, comparison_source)
    balance_comparison.to_csv(paths.table_directory / "balance_replication_comparison.csv", index=False)

    # Estimate recidivism effects under baseline, adjusted, kernel, and polynomial specifications.
    recidivism_results = estimate_recidivism_specifications(data)
    recidivism_results.to_csv(paths.table_directory / "recidivism_rd_results.csv", index=False)

    # Estimate sensitivity to local-window width and kernel choice.
    bandwidth_results = estimate_bandwidth_sensitivity(data)
    bandwidth_results.to_csv(paths.table_directory / "rd_bandwidth_sensitivity.csv", index=False)

    # Export side-specific sample sizes and raw recidivism rates near each cutoff.
    sample_diagnostics = rd_sample_diagnostics(data)
    sample_diagnostics.to_csv(paths.table_directory / "rd_sample_diagnostics.csv", index=False)

    # Estimate and export the exploratory discrete-running-variable density diagnostic.
    density_results, density_bins = exploratory_density_diagnostic(data)
    density_results.to_csv(paths.table_directory / "density_diagnostic.csv", index=False)

    # Estimate the primary cross-fitted ML-assisted RD that preserves cutoff identification.
    ml_rd_results, ml_nuisance_diagnostics = estimate_ml_residualized_rd(data)
    ml_rd_results.to_csv(paths.table_directory / "causal_ml_average_effects.csv", index=False)
    ml_nuisance_diagnostics.to_csv(paths.table_directory / "causal_ml_diagnostics.csv", index=False)

    # Estimate exploratory conditional discontinuities and their BAC-bin bootstrap uncertainty.
    (
        exploratory_average_results,
        exploratory_subgroup_results,
        exploratory_fold_diagnostics,
        seed_sensitivity,
        cate_data_by_cutoff,
        importance_by_cutoff,
    ) = estimate_exploratory_heterogeneity(data)
    exploratory_average_results.to_csv(
        paths.table_directory / "causal_ml_exploratory_average_effects.csv",
        index=False,
    )
    exploratory_subgroup_results.to_csv(
        paths.table_directory / "causal_ml_subgroup_effects.csv",
        index=False,
    )
    exploratory_fold_diagnostics.to_csv(
        paths.table_directory / "causal_ml_crossfit_diagnostics.csv",
        index=False,
    )
    seed_sensitivity.to_csv(
        paths.table_directory / "causal_ml_seed_sensitivity.csv",
        index=False,
    )

    # Combine transparent RD benchmarks and ML estimates into one comparison table.
    estimator_comparison = comparison_table(
        recidivism_results,
        ml_rd_results,
        exploratory_average_results,
    )
    estimator_comparison.to_csv(
        paths.table_directory / "causal_estimator_comparison.csv",
        index=False,
    )

    # Generate RD, sensitivity, density, heterogeneity, and feature-importance figures for each cutoff.
    for cutoff in THRESHOLDS.values():
        cutoff_label = f"{int(round(cutoff * 1000)):03d}"
        plot_recidivism_rd(
            data,
            cutoff,
            paths.figure_directory / f"recidivism_rd_bac_{cutoff_label}.png",
        )
        plot_bandwidth_sensitivity(
            bandwidth_results,
            cutoff,
            paths.figure_directory / f"rd_bandwidth_sensitivity_bac_{cutoff_label}.png",
        )
        plot_density_bins(
            density_bins[cutoff],
            cutoff,
            paths.figure_directory / f"density_bac_{cutoff_label}.png",
        )
        plot_cate_distribution(
            cate_data_by_cutoff[cutoff],
            cutoff,
            paths.figure_directory / f"cate_distribution_bac_{cutoff_label}.png",
        )
        plot_sorted_cates(
            cate_data_by_cutoff[cutoff],
            cutoff,
            paths.figure_directory / f"cate_sorted_bac_{cutoff_label}.png",
        )
        plot_subgroup_effects(
            exploratory_subgroup_results,
            cutoff,
            paths.figure_directory / f"cate_subgroups_bac_{cutoff_label}.png",
        )
        plot_feature_importance(
            importance_by_cutoff[cutoff],
            cutoff,
            paths.figure_directory / f"predictive_feature_importance_bac_{cutoff_label}.png",
        )
        importance_by_cutoff[cutoff].to_csv(
            paths.table_directory / f"predictive_feature_importance_bac_{cutoff_label}.csv",
            index=False,
        )

    # Save all specification and random-seed choices in a machine-readable metadata file.
    save_json_metadata(paths)

    # Write an execution summary generated directly from the estimated results.
    write_execution_summary(
        paths,
        balance_comparison,
        recidivism_results,
        ml_rd_results,
        exploratory_average_results,
        density_results,
    )

    # Print the central estimator comparison so a completed run is visible in the console.
    print("\nMain causal estimator comparison:")
    print(estimator_comparison.round(6).to_string(index=False))

    # Confirm the final output directories to make reproduction straightforward.
    print(f"\nTables saved to: {paths.table_directory}")
    print(f"Figures saved to: {paths.figure_directory}")
    print("Analysis completed successfully.")


# Execute the entire workflow when the file is run as a script or as the final notebook cell.
if __name__ == "__main__":
    main()

Project root: /mnt/data/hansen_project_work/hansen-drunk-driving-replication-main
    software version
      Python  3.13.5
      pandas   2.2.3
       numpy   2.3.5
 statsmodels  0.14.6
scikit-learn   1.8.0
    LightGBM   4.6.0
  matplotlib  3.10.8



Sample construction audit:
                           step  rows_remaining  rows_removed
                    Raw dataset          214558             0
Drop missing required variables          214558             0
    Restrict years to 1999-2007          214558             0
    Restrict age to 21 or older          214558             0



Main causal estimator comparison:
          threshold_name  cutoff                                                 method  estimate  standard_error  ci_lower  ci_upper  p_value  observations  observations_below  observations_at_or_above
           DUI threshold    0.08                               Baseline local linear RD -0.021845        0.004175 -0.030028 -0.013663 0.000000         93899               19679                     74220
           DUI threshold    0.08                     Covariate-adjusted local linear RD -0.022163        0.004142 -0.030281 -0.014046 0.000000         93899               19679                     74220
Aggravated DUI threshold    0.15                               Baseline local linear RD -0.007486        0.003500 -0.014345 -0.000626 0.032446        139033               79418                     59615
Aggravated DUI threshold    0.15                     Covariate-adjusted local linear RD -0.007282        0.003459 -0.014061 -0.000503 0.035256        139